# 🌲 Random Forests
**ISLP Chapter 8 · Data Pattern Recognition for the Rest of Us**

> Random Forests fix the main weakness of single decision trees — high variance — by averaging hundreds of trees, each trained on a random subset of data and features.

### Dataset
**IBM HR Analytics — Employee Attrition**
Predict which employees are likely to leave the company. Real HR data with 35 features including salary, job satisfaction, overtime, and years at company. Directly applicable to workforce planning and retention strategy.

### Time: ~60 minutes

In [ ]:
# IBM HR Analytics — Employee Attrition dataset
# Source: IBM Watson Analytics sample dataset (widely used in HR analytics)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv'
    attrition = pd.read_csv(url)
    print("Loaded IBM HR dataset from source")
except:
    # Reconstruct synthetically with realistic HR properties
    np.random.seed(42)
    n = 1470
    age = np.random.randint(18, 60, n)
    monthly_income = np.random.lognormal(8.5, 0.5, n).astype(int).clip(1000, 20000)
    job_satisfaction = np.random.randint(1, 5, n)
    overtime = np.random.binomial(1, 0.28, n)
    years_at_company = np.random.randint(0, 35, n)
    work_life_balance = np.random.randint(1, 5, n)
    distance_from_home = np.random.randint(1, 30, n)
    num_companies_worked = np.random.randint(0, 10, n)
    department = np.random.choice(['Sales','R&D','HR'], n, p=[0.3, 0.6, 0.1])

    # Attrition driven by: overtime, low satisfaction, low income, early career
    log_odds = (-1.5
                + 1.2 * overtime
                - 0.3 * job_satisfaction
                - 0.0001 * monthly_income
                + 0.3 * (years_at_company < 3).astype(float)
                + 0.1 * distance_from_home / 10
                + 0.2 * num_companies_worked / 5)
    prob_leave = 1 / (1 + np.exp(-log_odds))
    attrition_flag = (np.random.uniform(0,1,n) < prob_leave).astype(int)

    attrition = pd.DataFrame({
        'Age': age, 'MonthlyIncome': monthly_income,
        'JobSatisfaction': job_satisfaction, 'OverTime': np.where(overtime,'Yes','No'),
        'YearsAtCompany': years_at_company, 'WorkLifeBalance': work_life_balance,
        'DistanceFromHome': distance_from_home, 'NumCompaniesWorked': num_companies_worked,
        'Department': department, 'Attrition': np.where(attrition_flag,'Yes','No')
    })
    print("Using synthetic IBM HR-like dataset")

# Encode target and categoricals
attrition['Attrition_num'] = (attrition['Attrition'] == 'Yes').astype(int)
cat_cols = attrition.select_dtypes('object').columns.drop('Attrition')
for c in cat_cols:
    attrition[c] = LabelEncoder().fit_transform(attrition[c].astype(str))

feature_cols = [c for c in attrition.columns if c not in ['Attrition','Attrition_num']]
X = attrition[feature_cols]
y = attrition['Attrition_num']

print(f"\nIBM HR Attrition: {attrition.shape[0]:,} employees, {len(feature_cols)} features")
print(f"Attrition rate: {y.mean():.1%}  (class imbalance — typical for HR data)")
print(f"\nTop features: {feature_cols[:8]}")

## 🌳 Part 1 — Why Random Forests Beat Single Trees

A single decision tree memorizes the training data (100% training accuracy) but generalizes poorly.
Random Forests train hundreds of trees on bootstrap samples, each considering only a random
subset of features at each split. The decorrelation between trees is what makes the average powerful.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)

# Compare single tree vs random forest
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_tr, y_tr)

rf = RandomForestClassifier(n_estimators=500, oob_score=True,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print(f"{'Metric':<28} {'Decision Tree':>14} {'Random Forest':>14}")
print("-" * 58)
for metric, t_val, rf_val in [
    ('Training accuracy',     tree.score(X_tr,y_tr),   rf.score(X_tr,y_tr)),
    ('Test accuracy',         tree.score(X_te,y_te),   rf.score(X_te,y_te)),
    ('Test AUC-ROC',          roc_auc_score(y_te, tree.predict_proba(X_te)[:,1]),
                              roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])),
    ('Generalization gap',    tree.score(X_tr,y_tr)-tree.score(X_te,y_te),
                              rf.score(X_tr,y_tr)-rf.score(X_te,y_te))]:
    print(f"  {metric:<26} {t_val:>14.3f} {rf_val:>14.3f}")
print(f"\n  OOB error (free CV estimate):  {'N/A':>14} {1-rf.oob_score_:>14.3f}")

In [ ]:
# How test error changes with number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_errors, test_aucs = [], []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_n.oob_score_)
    test_aucs.append(roc_auc_score(y_te, rf_n.predict_proba(X_te)[:,1]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(n_trees_range, oob_errors, 's--', color='#e85d20', lw=2, label='OOB error')
axes[0].set_xlabel('Number of Trees'); axes[0].set_ylabel('Error Rate')
axes[0].set_title('OOB Error Stabilizes Around 100-200 Trees')
axes[0].set_xscale('log'); axes[0].legend()

axes[1].plot(n_trees_range, test_aucs, 'o-', color='#1e5fa8', lw=2, label='Test AUC-ROC')
axes[1].set_xlabel('Number of Trees'); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC vs Number of Trees\n(attrition prediction — IBM HR data)')
axes[1].set_xscale('log'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Which factors drive employee attrition? — Variable Importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e85d20' if v > importances.quantile(0.75) else '#1e5fa8'
          for v in importances]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('What Drives Employee Attrition?\n'
             'Random Forest Variable Importance (orange = top quartile)')
plt.tight_layout(); plt.show()

top3 = importances.tail(3).index.tolist()[::-1]
print(f"\n\u2714 Top 3 attrition drivers: {', '.join(top3)}")
print("  These are the levers HR should focus on for retention strategy")

In [ ]:
# ROC curve + business interpretation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

RocCurveDisplay.from_estimator(rf, X_te, y_te, ax=axes[0],
    name=f'Random Forest (AUC={roc_auc_score(y_te,rf.predict_proba(X_te)[:,1]):.3f})')
RocCurveDisplay.from_estimator(tree, X_te, y_te, ax=axes[0],
    name='Single Tree', color='#e85d20', alpha=0.7)
axes[0].set_title('ROC Curve — Attrition Prediction')

# Score distribution by actual outcome
rf_scores = rf.predict_proba(X_te)[:,1]
axes[1].hist(rf_scores[y_te==0], bins=30, alpha=0.6, color='#1e5fa8',
             density=True, label='Stayed')
axes[1].hist(rf_scores[y_te==1], bins=30, alpha=0.6, color='#e85d20',
             density=True, label='Left')
axes[1].axvline(0.5, color='black', lw=1.5, ls='--', label='Default threshold')
axes[1].set_xlabel('Predicted Attrition Probability')
axes[1].set_title('Score Distribution\n(good separation = useful model)')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# @title 📝 Quiz — Random Forests
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key difference between bagging and Random Forests?
# @markdown **Q2:** Why does OOB error give a valid estimate of test error?
# @markdown **Q3:** What does variable importance measure in a Random Forest?
# @markdown **Q4:** If HR wants to reduce attrition, which top feature would you investigate first and why?
# @markdown **Q5:** Why does a single decision tree have 100% training accuracy but Random Forest does not?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="Random Forests"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*